# Séance 3 — Nettoyage et qualité des données (corrigé)

**Durée pratique : 3 h 30** &nbsp;·&nbsp; Ateliers 3.1 à 3.4

## Ce que vous saurez faire à la fin

- classer les valeurs manquantes et choisir un traitement adapté à chaque colonne ;
- détecter les valeurs aberrantes par écart interquartile et par score z, et arbitrer ;
- normaliser des chaînes de caractères pour éliminer les faux doublons ;
- écrire une fonction de nettoyage idempotente qui journalise ce qu'elle modifie.

> **Le principe directeur de la séance.** Une décision de nettoyage n'est jamais neutre :
> elle change les résultats en aval. Chaque choix doit donc être **explicite, justifié et
> tracé**. Un pipeline qui supprime 4 % des lignes sans le dire est un pipeline dangereux.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

In [ ]:
def parse_price(series):
    """Convertit une colonne de prix mixte (nombre ou texte '123,45 EUR') en float."""
    text = series.astype(str).str.replace(' EUR', '', regex=False)
    return pd.to_numeric(text.str.replace(',', '.', regex=False).str.strip(),
                         errors='coerce')


sales = pd.read_csv(RAW_DIR / 'ventes_brutes.csv')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
sales['prix_unitaire'] = parse_price(sales['prix_unitaire'])

print(sales.shape, customers.shape)

---
## Atelier 3.1 — Cartographier les valeurs manquantes (60 min)

### Partie guidée

Avant de choisir un traitement, il faut savoir **pourquoi** la valeur est absente.
La typologie classique distingue trois cas :

| Type | Signification | Conséquence |
|---|---|---|
| MCAR | absence totalement aléatoire | supprimer ne biaise pas, on perd seulement de la puissance |
| MAR | absence expliquée par d'autres colonnes | imputation conditionnelle possible et souhaitable |
| MNAR | absence liée à la valeur elle-même | tout traitement biaise ; l'absence doit devenir une information |

On ne peut pas prouver le type à partir des seules données. On l'argumente à partir de la
connaissance du processus qui les a produites.

In [ ]:
missing_summary = pd.DataFrame({
    'nb_manquants': sales.isna().sum(),
    'taux_pct': (sales.isna().mean() * 100).round(2),
})
missing_summary[missing_summary['nb_manquants'] > 0].sort_values('taux_pct', ascending=False)

In [ ]:
# Les remises manquantes le sont-elles au hasard ? Comparons les lignes concernées
# au reste du jeu sur quelques variables.
has_no_discount = sales['remise_pct'].isna()

comparison = pd.DataFrame({
    'remise manquante': sales[has_no_discount][['quantite', 'prix_unitaire']].mean(),
    'remise renseignée': sales[~has_no_discount][['quantite', 'prix_unitaire']].mean(),
}).round(2)
print(comparison)
print()
print('Répartition des statuts, remise manquante :')
print(sales[has_no_discount]['statut'].value_counts(normalize=True).round(3))
print('Répartition des statuts, remise renseignée :')
print(sales[~has_no_discount]['statut'].value_counts(normalize=True).round(3))

**Lecture.** Si les deux profils sont semblables, l'hypothèse MCAR est plausible et une
imputation simple par zéro se défend (pas de remise saisie = pas de remise accordée).
S'ils diffèrent nettement, l'absence porte de l'information : créer un indicateur
`remise_absente` devient préférable à une imputation silencieuse.

### Partie autonome

In [ ]:
# Q1. Même analyse pour la colonne 'age' de customers : les âges manquants se
#     concentrent-ils sur certains segments ou certaines villes ?

has_no_age = customers['age'].isna()
print(f"Âges manquants : {has_no_age.sum()} sur {len(customers)} ({has_no_age.mean():.2%})")

for column in ['segment', 'ville']:
    rates = (customers.assign(age_manquant=has_no_age)
             .groupby(column)['age_manquant']
             .agg(nb_clients='size', nb_manquants='sum', taux='mean')
             .sort_values('taux', ascending=False))
    rates['taux'] = (rates['taux'] * 100).round(1)
    print(f'\nTaux d\'âge manquant par {column} (%) :')
    print(rates.to_string())

# Comparaison de la répartition des segments avec / sans âge
print('\nRépartition des segments, âge manquant :')
print(customers[has_no_age]['segment'].value_counts(normalize=True).round(3).to_string())
print('Répartition des segments, âge renseigné :')
print(customers[~has_no_age]['segment'].value_counts(normalize=True).round(3).to_string())

# Lecture : si le taux de manquants varie fortement d'un segment à l'autre (un segment à
# 30 %, les autres à 2 %), l'absence est liée au segment -> MAR : on impute la médiane
# PAR SEGMENT. Si les taux sont homogènes, l'hypothèse MCAR tient et la médiane globale suffit.

In [ ]:
# Q2. Rédiger une stratégie colonne par colonne, sous forme de dictionnaire.
#     Valeurs autorisées : 'drop_row', 'fill_zero', 'fill_median',
#     'fill_median_by_group', 'add_indicator', 'keep_as_is'.
#     La justification est OBLIGATOIRE : c'est elle qui sera évaluée, pas le choix lui-même.

missing_value_strategy = {
    'date_commande': ('drop_row',
                      'une commande sans date est inexploitable pour toute analyse temporelle'),
    'remise_pct':    ('fill_zero',
                      'les lignes sans remise ont le même profil (quantité, prix, statut) que '
                      'les autres : absence MCAR plausible ; métier : pas de remise saisie = '
                      'pas de remise accordée'),
    'age':           ('fill_median_by_group',
                      "l'absence varie selon le segment (MAR) : la médiane du segment est plus "
                      "juste que la médiane globale ; médiane plutôt que moyenne car l'âge est "
                      'asymétrique et sensible aux valeurs extrêmes'),
    'cout_achat':    ('keep_as_is',
                      "donnée de référentiel produit : inventer un coût fausserait la marge ; "
                      "on laisse NaN, la marge reste inconnue pour ces produits et on le signale"),
}

for column, (action, rationale) in missing_value_strategy.items():
    print(f'{column:<16} {action:<24} {rationale}')

---
## Atelier 3.2 — Valeurs aberrantes : deux méthodes, deux verdicts (55 min)

### Partie guidée

In [ ]:
def iqr_bounds(series, factor=1.5):
    """Bornes de Tukey : robustes, car fondées sur des quantiles."""
    q1, q3 = series.quantile([0.25, 0.75])
    spread = q3 - q1
    return q1 - factor * spread, q3 + factor * spread


def zscore_bounds(series, n_std=3):
    """Bornes par score z : sensibles, car moyenne et écart type sont eux-mêmes
    tirés vers le haut par les valeurs extrêmes."""
    mean, std = series.mean(), series.std()
    return mean - n_std * std, mean + n_std * std


quantity = sales['quantite']
for label, (lower, upper) in [('IQR    ', iqr_bounds(quantity)),
                              ('score z', zscore_bounds(quantity))]:
    n_flagged = ((quantity < lower) | (quantity > upper)).sum()
    print(f'{label} bornes [{lower:8.2f} ; {upper:8.2f}]  ->  {n_flagged:>4} valeurs signalées')

Les deux méthodes ne s'accordent pas, et l'écart est instructif. Le score z est calculé à
partir de la moyenne et de l'écart type, deux statistiques que les valeurs extrêmes
**déforment elles-mêmes**. Quelques quantités à 1 000 suffisent à gonfler l'écart type au
point que ces mêmes valeurs passent sous le seuil de détection. L'IQR, fondé sur des
quantiles, ne souffre pas de cet effet.

In [ ]:
# Regarder les valeurs concernées avant de décider quoi que ce soit
print('Quantités les plus élevées :')
print(sales.nlargest(8, 'quantite')[['id_commande', 'quantite', 'prix_unitaire', 'statut']])
print()
print('Quantités négatives :', (sales['quantite'] < 0).sum())
print(sales[sales['quantite'] < 0][['id_commande', 'quantite', 'statut']].head())

**Deux problèmes de nature différente sont ici confondus.** Les quantités négatives sont
*impossibles* : c'est une erreur de saisie ou de signe, elle se corrige ou se supprime sans
état d'âme. Les quantités à 800 sont *improbables* mais pas impossibles : il peut s'agir de
commandes professionnelles légitimes. Les traiter de la même façon serait une faute
méthodologique.

### Partie autonome

In [ ]:
# Q3. Écrire une fonction qui, pour une colonne donnée, renvoie un DataFrame comparant
#     les deux méthodes : nb détectés, part en %, valeur min et max des points signalés.

def compare_outlier_methods(df, column):
    values = df[column].dropna()
    rows = []
    for method, (lower, upper) in [('IQR', iqr_bounds(values)),
                                   ('score z', zscore_bounds(values))]:
        flagged = values[(values < lower) | (values > upper)]
        rows.append({
            'methode': method,
            'borne_basse': round(lower, 2),
            'borne_haute': round(upper, 2),
            'nb_detectes': len(flagged),
            'part_pct': round(100 * len(flagged) / len(values), 2),
            'min_signale': flagged.min() if len(flagged) else np.nan,
            'max_signale': flagged.max() if len(flagged) else np.nan,
        })
    return pd.DataFrame(rows).set_index('methode')


print(compare_outlier_methods(sales, 'quantite'), end='\n\n')
print(compare_outlier_methods(sales, 'prix_unitaire'))

In [ ]:
# Q4. Visualiser. Un boxplot par statut pour la quantité, en échelle logarithmique
#     sur l'axe des ordonnées (sans quoi les valeurs extrêmes écrasent tout le reste).

import matplotlib.pyplot as plt

# Le log n'est pas défini pour les valeurs <= 0 : on ne trace que les quantités
# positives (les négatives sont traitées à la question suivante).
plot_data = sales.loc[sales['quantite'] > 0, ['quantite', 'statut']].copy()
plot_data['statut'] = plot_data['statut'].str.strip().str.lower()

statuses = sorted(plot_data['statut'].dropna().unique())
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([plot_data.loc[plot_data['statut'] == s, 'quantite'] for s in statuses],
           tick_labels=statuses)
ax.set_yscale('log')
ax.set_ylabel('quantité (échelle log)')
ax.set_title('Quantité commandée par statut')
plt.tight_layout()
plt.show()

print('Quantités <= 0 exclues du graphique :', (sales['quantite'] <= 0).sum())

In [ ]:
# Q5. Appliquer un traitement différencié et JOURNALISER ce qui est fait :
#       - quantités négatives : SUPPRIMÉES. Une quantité négative est impossible
#         (erreur de saisie ou de signe). On ne peut pas savoir avec certitude si la
#         valeur absolue est la bonne : corriger reviendrait à inventer une donnée.
#       - quantités au-delà de la borne haute IQR : CONSERVÉES et SIGNALÉES par une
#         colonne quantite_atypique. Elles sont improbables mais possibles (commandes
#         professionnelles) : les supprimer ferait disparaître du vrai chiffre d'affaires.
#     La fonction renvoie (df_traité, log) où log est un dict des comptes.

def handle_quantity_outliers(df):
    df = df.copy()
    log = {'lignes_initiales': len(df)}

    negative = df['quantite'] < 0
    log['negatives_supprimees'] = int(negative.sum())
    log['negatives_par_statut'] = (df.loc[negative, 'statut'].str.strip().str.lower()
                                   .value_counts().to_dict())
    df = df[~negative]

    # Bornes calculées APRÈS retrait des négatives, qui fausseraient le premier quartile
    _, upper = iqr_bounds(df['quantite'])
    df['quantite_atypique'] = df['quantite'] > upper
    log['borne_haute_iqr'] = round(float(upper), 2)
    log['atypiques_signalees'] = int(df['quantite_atypique'].sum())
    log['ca_atypiques_pct'] = round(
        100 * (df.loc[df['quantite_atypique'], 'quantite']
               * df.loc[df['quantite_atypique'], 'prix_unitaire']).sum()
        / (df['quantite'] * df['prix_unitaire']).sum(), 2)

    log['lignes_finales'] = len(df)
    return df, log


sales_fixed, quantity_log = handle_quantity_outliers(sales)
for key, value in quantity_log.items():
    print(f'  {key:<24} {value}')

---
## Atelier 3.3 — Doublons et incohérences textuelles (45 min)

### Partie guidée

In [ ]:
print('Doublons stricts (toutes colonnes identiques) :', sales.duplicated().sum())
print('Doublons sur id_commande seul                 :', sales.duplicated('id_commande').sum())

Les deux comptes diffèrent : certaines lignes partagent un identifiant de commande sans
être identiques. Avant de supprimer, il faut regarder ce qui les distingue.

In [ ]:
repeated_ids = sales[sales.duplicated('id_commande', keep=False)]['id_commande'].unique()[:2]
sales[sales['id_commande'].isin(repeated_ids)].sort_values('id_commande')

In [ ]:
# Faux doublons textuels : la même ville écrite de plusieurs façons
print(sales['ville_livraison'].value_counts().head(12))

In [ ]:
def normalize_text(series):
    """Casse, espaces et accents : trois sources de faux niveaux."""
    text = series.astype(str).str.strip().str.lower()
    text = text.str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('utf-8')
    return text.str.replace(r'\s+', ' ', regex=True)


print('Modalités avant normalisation :', sales['ville_livraison'].nunique())
print('Modalités après normalisation :', normalize_text(sales['ville_livraison']).nunique())

### Le piège : ne jamais normaliser les colonnes identifiantes

La tentation est grande d'écrire `for column in df.select_dtypes('object'): df[column] = normalize_text(df[column])`
et d'en finir. **C'est une erreur qui casse silencieusement tout le pipeline.** Les colonnes
`id_client`, `id_produit`, `id_magasin` sont du texte pour pandas, mais ce sont des **clés**.
Les passer en minuscules transforme `P0027` en `p0027`, et la jointure de la séance 6 ne
trouvera plus aucune correspondance : vous obtiendrez un jeu entièrement vide, sans la
moindre erreur levée.

Déclarez donc explicitement les colonnes à normaliser, plutôt que de les déduire du type.

In [ ]:
TEXT_COLUMNS = ['canal', 'statut', 'ville_livraison']          # descriptives uniquement
KEY_COLUMNS = ['id_commande', 'id_client', 'id_produit', 'id_magasin']  # à préserver

trial = sales.copy()
for column in TEXT_COLUMNS:
    trial[column] = normalize_text(trial[column])

print('Clés intactes            :', trial['id_produit'].str.isupper().all())
print('Descriptives normalisées :', trial['canal'].nunique(), 'modalités')

### Partie autonome

In [ ]:
# Q6. Normaliser toutes les colonnes de TEXT_COLUMNS, puis recompter les doublons stricts.
#     Le compte augmente-t-il ? Expliquez pourquoi dans un commentaire.

normalized = sales.copy()
for column in TEXT_COLUMNS:
    normalized[column] = normalize_text(normalized[column])

print('Doublons stricts avant normalisation :', sales.duplicated().sum())
print('Doublons stricts après normalisation :', normalized.duplicated().sum())

# Le compte augmente dès que la source contient des variantes d'écriture : deux lignes
# qui ne différaient QUE par la casse, un espace ou
# un accent ('Web' / ' web', 'Évry' / 'evry') étaient vues comme différentes. Une fois
# normalisées, elles deviennent strictement identiques et apparaissent comme doublons.
# Conséquence pratique : on normalise d'abord, on dédoublonne ensuite — c'est l'ordre
# retenu dans clean_sales.

In [ ]:
# Q7. Dans customers, identifier les doublons MÉTIER : même ville, même segment,
#     même date d'inscription et même âge, mais identifiant différent.
#     Combien de couples trouvez-vous ? Que proposez-vous d'en faire ?

BUSINESS_KEY = ['ville', 'segment', 'date_inscription', 'age']

# On écarte les âges manquants : deux NaN ne prouvent pas que deux clients sont identiques
candidates = customers.dropna(subset=BUSINESS_KEY)
groups = (candidates.groupby(BUSINESS_KEY)['id_client']
          .agg(nb_ids='nunique', ids=lambda s: sorted(s.unique())))
groups = groups[groups['nb_ids'] > 1]

# Un groupe de n identifiants forme n*(n-1)/2 couples
n_pairs = int((groups['nb_ids'] * (groups['nb_ids'] - 1) // 2).sum())
print(f'Groupes suspects : {len(groups)} | couples de doublons métier : {n_pairs}')
print(groups.head(10).to_string())

# Proposition : NE PAS fusionner automatiquement. Ces quatre attributs peuvent coïncider
# par hasard (même âge, même ville, inscrits le même jour), surtout dans une grande ville.
# On ajoute une colonne doublon_potentiel et un identifiant de groupe, on signale la liste
# au métier, et on ne fusionne qu'avec un critère discriminant (email, téléphone, nom).
customers['doublon_potentiel'] = customers.set_index(BUSINESS_KEY).index.isin(groups.index)
print('\nClients marqués doublon_potentiel :', customers['doublon_potentiel'].sum())

---
## Atelier 3.4 — Un module de nettoyage idempotent (50 min)

**Idempotent** signifie que réappliquer la fonction à son propre résultat ne change plus rien.
C'est une propriété essentielle : un pipeline relancé deux fois par erreur ne doit pas
produire un jeu de données différent.

In [ ]:
TEXT_COLUMNS = ['canal', 'statut', 'ville_livraison']                 # descriptives
KEY_COLUMNS = ['id_commande', 'id_client', 'id_produit', 'id_magasin']  # jamais modifiées


def parse_price(series):
    """Convertit une colonne de prix mixte (nombre ou texte '123,45 EUR') en float."""
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float)
    text = series.astype(str).str.replace(' EUR', '', regex=False)
    return pd.to_numeric(text.str.replace(',', '.', regex=False).str.strip(),
                         errors='coerce')


def normalize_text_safe(series):
    """normalize_text qui laisse les valeurs manquantes manquantes (pas de 'nan' texte)."""
    text = series.astype(str).str.strip().str.lower()
    text = text.str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('utf-8')
    text = text.str.replace(r'\s+', ' ', regex=True)
    return text.where(series.notna())


def parse_mixed_dates(series):
    """Dates hétérogènes -> datetime.

    Les formats « année en tête » (2024-03-05) sont lus SANS dayfirst, les autres
    (05/03/2024) AVEC : avec format='mixed', dayfirst=True inverserait aussi le jour
    et le mois des dates ISO.
    """
    if pd.api.types.is_datetime64_any_dtype(series):
        return series
    text = series.astype('string').str.strip()
    year_first = text.str.match(r'^\d{4}[-/.]', na=False)
    parsed = pd.concat([
        pd.to_datetime(text[year_first], format='mixed', errors='coerce'),
        pd.to_datetime(text[~year_first], format='mixed', dayfirst=True, errors='coerce'),
    ])
    return pd.to_datetime(parsed.reindex(series.index))


def clean_sales(df, drop_negative_quantities=True, max_quantity=None, verbose=True):
    """Nettoie le jeu de ventes et renvoie (df_propre, log).

    Étapes, dans cet ordre :
      1. copier le DataFrame reçu (ne jamais modifier l'entrée) ;
      2. convertir prix_unitaire en numérique ;
      3. normaliser les colonnes texte DESCRIPTIVES (jamais les identifiants) ;
      4. supprimer les doublons stricts ;
      5. convertir date_commande en datetime et supprimer les lignes sans date ;
      6. appliquer la stratégie retenue pour remise_pct (fill_zero) ;
      7. traiter les quantités selon les paramètres ;
      8. calculer la colonne montant.

    Le log recense, pour chaque étape, le nombre de lignes ou de valeurs touchées.
    Chaque étape est idempotente : un second passage ne change plus rien.
    """
    log = {'lignes_initiales': len(df)}
    df = df.copy()

    # 2. Prix
    was_numeric = pd.api.types.is_numeric_dtype(df['prix_unitaire'])
    df['prix_unitaire'] = parse_price(df['prix_unitaire'])
    log['prix_non_convertibles'] = int(df['prix_unitaire'].isna().sum())
    log['prix_deja_numeriques'] = bool(was_numeric)

    # 3. Texte descriptif uniquement
    for column in TEXT_COLUMNS:
        before = df[column].nunique()
        df[column] = normalize_text_safe(df[column])
        log[f'modalites_{column}'] = f'{before} -> {df[column].nunique()}'

    # 4. Doublons stricts (APRÈS normalisation : ' Web' et 'web' doivent coïncider)
    n_before = len(df)
    df = df.drop_duplicates()
    log['doublons_stricts_supprimes'] = n_before - len(df)

    # 5. Dates
    df['date_commande'] = parse_mixed_dates(df['date_commande'])
    n_before = len(df)
    df = df.dropna(subset=['date_commande'])
    log['lignes_sans_date_supprimees'] = n_before - len(df)

    # 6. Remise : MCAR plausible (profils identiques), absence = pas de remise
    log['remises_imputees_a_zero'] = int(df['remise_pct'].isna().sum())
    df['remise_pct'] = df['remise_pct'].fillna(0)

    # 7. Quantités
    negative = df['quantite'] < 0
    if drop_negative_quantities:
        df = df[~negative]
        log['quantites_negatives_supprimees'] = int(negative.sum())
    else:
        df.loc[negative, 'quantite'] = df.loc[negative, 'quantite'].abs()
        log['quantites_negatives_corrigees'] = int(negative.sum())
    if max_quantity is not None:
        too_large = df['quantite'] > max_quantity
        df = df[~too_large]
        log[f'quantites_sup_{max_quantity}_supprimees'] = int(too_large.sum())

    # 8. Montant (brut, avant remise)
    df['montant'] = (df['quantite'] * df['prix_unitaire']).round(2)

    df = df.reset_index(drop=True)
    log['lignes_finales'] = len(df)
    log['lignes_supprimees_pct'] = round(100 * (1 - len(df) / log['lignes_initiales']), 2)
    if verbose:
        for key, value in log.items():
            print(f'  {key:<34} {value}')
    return df, log


clean_sales_df, cleaning_log = clean_sales(pd.read_csv(RAW_DIR / 'ventes_brutes.csv'))

In [ ]:
# Q8. Vérifier l'idempotence : un second passage ne doit rien changer.

second_pass, _ = clean_sales(clean_sales_df, verbose=False)

assert len(second_pass) == len(clean_sales_df), "la fonction n'est pas idempotente"
assert second_pass.equals(clean_sales_df), 'le second passage modifie encore des valeurs'
print('OK — fonction idempotente')

In [ ]:
# Export du résultat et du journal, pour la séance 4.

import json

save_dataset(clean_sales_df, 'ventes_nettoyees')

with open(PROCESSED_DIR / 'journal_nettoyage.json', 'w', encoding='utf-8') as file:
    json.dump(cleaning_log, file, indent=2, ensure_ascii=False, default=str)

kept_ratio = len(clean_sales_df) / cleaning_log['lignes_initiales']
print('Lignes conservées :', len(clean_sales_df), f'({kept_ratio:.1%} du brut)')

---
## Livrable de la séance

- `src/cleaning.py` contenant `clean_sales`, importable et testée ;
- `data/processed/ventes_nettoyees.parquet` ;
- `data/processed/journal_nettoyage.json` ;
- le tableau des stratégies (question 2) avec sa justification écrite, colonne par colonne.

**Ce tableau sera repris tel quel dans votre note méthodologique de projet et vous sera
demandé en soutenance.** Attendez-vous à la question : *pourquoi la médiane plutôt que la
moyenne sur cette colonne ?*